# Deep learning on images

## Load modules from repo

In [74]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [75]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [76]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.image import get_image_path, load_image, get_image_features_with_hash, get_image_md5_hash
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model, get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

In [77]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.image)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)

new


<module 'src.models.on_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_images/deep_learning.py'>

## Load tensorflow

In [12]:
import tensorflow as tf

In [13]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [14]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [15]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')

In [16]:
# Class distribution
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration

In [30]:
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = False  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

BATCH_SIZE = 32

RANDOM_SEED = 42

## Preprocessing

In [62]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [ ]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [64]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024700
min       0.008980
25%       0.018696
50%       0.031797
75%       0.054541
max       0.122185
Name: proportion, dtype: float64

In [65]:
y_train.value_counts().describe()

count     27.000000
mean     251.592593
std      167.786325
min       61.000000
25%      127.000000
50%      216.000000
75%      370.500000
max      830.000000
Name: count, dtype: float64

In [66]:
print(X_train.shape)

(6793, 31)


In [67]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [ ]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, shuffle=True, BATCH_SIZE = BATCH_SIZE)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [69]:
new_preprocessors

{}

In [70]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [ ]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE)

## Model

### Create model

In [72]:
# model = define_model(embedding_dim=16, n_cols_tabular=n_cols_tabular, num_classes=27)

### Load a saved model

In [50]:
from tensorflow import keras
model = keras.models.load_model(artifacts_folder / 'best_model-1.keras')

I0000 00:00:1759324228.501386    9261 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


### Load saved weights

In [74]:
# model.load_weights(artifacts_folder / 'best_model-1.h5')

### Summary

In [75]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 500, 500,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 500, 500,  │          0 │ image_input[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 500, 500,  │          0 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 250, 250,  │        864 │ normalization[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 250, 250,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 250, 250,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 250, 250,  │      4,608 │ stem_activation[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 250, 250,  │         64 │ block1a_project_… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_ac… │ (None, 250, 250,  │          0 │ block1a_project_… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_conv │ (None, 125, 125,  │      9,216 │ block1a_project_… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_bn   │ (None, 125, 125,  │        256 │ block2a_expand_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_expand_act… │ (None, 125, 125,  │          0 │ block2a_expand_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_co… │ (None, 125, 125,  │      2,048 │ block2a_expand_a… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2a_project_bn  │ (None, 125, 125,  │        128 │ block2a_project_… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_conv │ (None, 125, 125,  │     36,864 │ block2a_project_… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_bn   │ (None, 125, 125,  │        512 │ block2b_expand_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2b_expand_act… │ (None, 125, 125,  │          0 │ block2b_expand_b

 Total params: 12,654,275 (48.27 MB)

 Trainable params: 2,244,987 (8.56 MB)

 Non-trainable params: 5,919,312 (22.58 MB)

 Optimizer params: 4,489,976 (17.13 MB)

## Training

### Callbacks

#### Saving

In [76]:
# Pick an available filename to save a model.
k=1
while True:
    new_location_for_saving_model = Path(artifacts_folder / f'best_model-{k}.keras')
    if not new_location_for_saving_model.exists():
        break
    k+=1
# new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{k}.h5')
new_location_for_saving_model


PosixPath('artifacts/on_images/deep_learning/v1/best_model-2.keras')

In [77]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [78]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

#### EarlyStopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la 'val_accuracy' ne s'améliore pas
# pendant 3 epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_accuracy', # La métrique à surveiller
    patience=3,             # Nombre d'epochs à attendre sans amélioration
    mode='max',             # On veut maximiser l'accuracy
    restore_best_weights=True # Très important : à la fin, le modèle aura les poids de son meilleur score
)

#### ReduceLROnPlateau

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On divise le LR par 5 (LR * 0.2)
    patience=2,         # Si val_loss stagne pendant 2 epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### Setup

In [ ]:
max_epochs=7  # TODO: try 50
learning_rate=0.001

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [save, early_stopping, reduce_lr]

In [85]:
if rebalance_with_weights:
    if small_train_sample:
        raise ValueError("Warning: rebalance_with_weights should not be used with small_train_sample.")
else:
    class_weights=None
    print('not using class_weights')

not using class_weights


### Training

In [86]:
raise Exception("Are you sure you want to launch training?")

Exception: Are you sure you want to launch training?

In [ ]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=max_epochs, callbacks=callbacks, class_weights=class_weights)
end = time.time()
total_minutes = (end - start)/60

Epoch 1/2


2025-09-30 16:46:53.921208: I external/local_xla/xla/service/service.cc:163] XLA service 0x70c3a4002890 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-09-30 16:46:53.921261: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-09-30 16:46:54.273560: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-09-30 16:46:55.820471: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-09-30 16:46:56.851699: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-30 16:46:56.

212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.2742 - loss: 2.5719

2025-09-30 16:47:53.037479: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:53.131945: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:53.705628: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:53.805287: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:47:54.427563: E external/local_xla/xla/stream_

213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.2747 - loss: 2.5702

2025-09-30 16:49:12.203358: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-09-30 16:49:19.919077: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:49:20.017448: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-09-30 16:49:20.824995: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

213/213 ━━━━━━━━━━━━━━━━━━━━ 159s 596ms/step - accuracy: 0.3694 - loss: 2.2063 - val_accuracy: 0.4941 - val_loss: 1.7053
Epoch 2/2
213/213 ━━━━━━━━━━━━━━━━━━━━ 89s 418ms/step - accuracy: 0.5073 - loss: 1.6425 - val_accuracy: 0.5381 - val_loss: 1.5457


## Evaluation

In [ ]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['val_accuracy'] = max(model_history.history['val_accuracy'])
tracker['actual_epochs']=len(model_history.history['val_accuracy'])
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

In [ ]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

2025-10-01 13:04:41.541075: I external/local_xla/xla/service/service.cc:163] XLA service 0x72063000f680 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-01 13:04:41.541095: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-01 13:04:41.651370: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-01 13:04:42.270033: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-01 13:04:42.384703: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-01 13:04:42.

530/531 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step

2025-10-01 13:06:02.566374: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-01 13:06:10.407402: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-01 13:06:10.505170: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-01 13:06:11.379103: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

531/531 ━━━━━━━━━━━━━━━━━━━━ 98s 146ms/step


array([[4.03293734e-03, 2.56265956e-03, 5.15742786e-03, ...,
        2.37797201e-02, 2.52174097e-03, 9.57679120e-04],
       [8.99143331e-03, 5.67056378e-03, 7.03551294e-03, ...,
        3.66787836e-02, 4.73827648e-04, 7.56713154e-04],
       [7.03595579e-04, 8.50657746e-03, 2.80403197e-02, ...,
        7.62895197e-02, 1.78870250e-04, 5.33306622e-04],
       ...,
       [1.10862255e-01, 3.16806766e-03, 9.94069414e-05, ...,
        3.82519065e-05, 8.05685580e-01, 7.40520190e-04],
       [2.45917425e-03, 5.45170344e-03, 1.05293337e-02, ...,
        9.68058780e-02, 3.29912378e-04, 3.43382213e-04],
       [4.25931485e-03, 1.35830184e-02, 1.53474398e-02, ...,
        2.53389850e-02, 2.27382500e-03, 3.12547141e-04]],
      shape=(16984, 27), dtype=float32)

In [97]:
y_test

81432    1301
44734    1560
59366    2583
36932    1160
69999    2403
         ... 
16142    1280
71018    2403
54062    2705
38340    2522
47310    2583
Name: prdtypecode, Length: 16984, dtype: int64

In [ ]:
from sklearn import metrics

y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [99]:
metrics.f1_score(y_test_class, y_pred_class, average='weighted')

0.4988805485858348

In [ ]:
print(metrics.classification_report(y_test_class, y_pred_class))

              precision    recall  f1-score   support

          10       0.45      0.37      0.41       623
          40       0.48      0.21      0.29       502
          50       0.44      0.15      0.22       336
          60       0.45      0.62      0.52       166
        1140       0.52      0.69      0.59       534
        1160       0.77      0.81      0.79       791
        1180       1.00      0.01      0.01       153
        1280       0.42      0.28      0.33       974
        1281       0.34      0.06      0.11       414
        1300       0.59      0.69      0.64      1009
        1301       1.00      0.01      0.01       161
        1302       0.39      0.07      0.12       498
        1320       0.41      0.37      0.39       648
        1560       0.48      0.59      0.53      1015
        1920       0.60      0.84      0.70       861
        1940       0.77      0.17      0.28       161
        2060       0.40      0.48      0.44       999
        2220       0.00    

In [112]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

,precision,recall,f1-score,support
10,0.450098,0.369181,0.405644,623.000000
40,0.483871,0.209163,0.292072,502.000000
50,0.441441,0.145833,0.219239,336.000000
60,0.445887,0.620482,0.518892,166.000000
1140,0.516174,0.687266,0.589558,534.000000
1160,0.765755,0.814159,0.789216,791.000000
1180,1.000000,0.006536,0.012987,153.000000
1280,0.421217,0.277207,0.334365,974.000000
1281,0.342105,0.062802,0.106122,414.000000
1300,0.591062,0.694747,0.638724,1009.000000


In [ ]:
# f1 is not as (linearly) correlated with support as I expected: 0.10
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.107428,0.183710,-0.014135
recall,0.107428,1.000000,0.959813,0.106023
f1-score,0.183710,0.959813,1.000000,0.099523
support,-0.014135,0.106023,0.099523,1.000000


In [121]:
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.539753,0.418221,0.406944,629.037037
std,0.219837,0.299595,0.238552,420.759339
min,0.000000,0.000000,0.000000,153.000000
25%,0.431329,0.156768,0.247375,310.000000
50%,0.483871,0.369181,0.405644,534.000000
75%,0.614348,0.682393,0.611664,953.500000
max,1.000000,0.911232,0.789216,2042.000000


In [115]:
report['f1-score'].iloc[:-3].describe()

count    27.000000
mean      0.406944
std       0.238552
min       0.000000
25%       0.247375
50%       0.405644
75%       0.611664
max       0.789216
Name: f1-score, dtype: float64

## Update tracker

In [70]:
tracker['comment']='Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks.'

In [71]:
to_track=['version','minutes_per_epoch','max_epochs','actual_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model)
tracker

{'version': 1,
 'rebalance_with_weights': False,
 'X_train.shape[0]': 67932,
 'BATCH_SIZE': 32,
 'minutes_per_epoch': 2.058333333333333,
 'max_epochs': 2,
 'actual_epochs': 2,
 'learning_rate': 0.001,
 'val_accuracy': 0.5381,
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16},
 'comment': 'Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks.'}

In [73]:
print(list(tracker))

['version', 'rebalance_with_weights', 'X_train.shape[0]', 'BATCH_SIZE', 'minutes_per_epoch', 'max_epochs', 'actual_epochs', 'learning_rate', 'val_accuracy', 'dense_layers_sizes', 'embedding_dims', 'comment']


In [57]:
log_file_path=artifacts_folder / 'experiments.parquet'
log_file_path

PosixPath('artifacts/on_images/deep_learning/v1/experiments.parquet')

In [79]:
log_experiment(tracker, log_file_path=log_file_path)

Creating parquet log.
Expérience version 1 sauvegardée dans artifacts/on_images/deep_learning/v1/experiments.parquet


In [81]:
pd.set_option('max_colwidth', None)

## Show tracking logs

In [82]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,version,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,dense_layers_sizes,embedding_dims,comment
0,1,False,67932,32,2.058333,2,2,0.001,0.5381,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
